# Starter-Notebook

Ziel: Erstelle ein synthetisches Dataset (Klassifikation oder Regression), führe eine schnelle EDA durch und trainiere eine einfache scikit-learn-Pipeline.

Anleitung: Führe die Zellen der Reihe nach aus. Du kannst die Parameter des Generators anpassen, um verschiedene Szenarien zu testen.

In [ ]:
# Basis-Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import classification_report, mean_squared_error

sns.set(style="whitegrid")
RANDOM_STATE = 42

In [ ]:
# Generator für synthetische Daten (Deutsch: Merkmal = 'merkmal_i', Ziel = 'ziel')
def generiere_synthetische_daten(art='klassifikation', n_samples=1000, n_features=10, rauschen=0.1, random_state=RANDOM_STATE):
    import sklearn.datasets as ds
    if art == 'klassifikation':
        X, y = ds.make_classification(n_samples=n_samples, n_features=n_features, n_informative=int(n_features/2), n_redundant=0, n_clusters_per_class=1, flip_y=rauschen, random_state=random_state)
    else:
        X, y = ds.make_regression(n_samples=n_samples, n_features=n_features, n_informative=int(n_features/2), noise=rauschen*10, random_state=random_state)
    spalten = [f'merkmal_{i}' for i in range(X.shape[1])]
    df = pd.DataFrame(X, columns=spalten)
    df['ziel'] = y
    return df

# Beispiel: Erzeuge ein Klassifikations-Dataset und speichere es als CSV (sichtbar auf GitHub)
df = generiere_synthetische_daten(art='klassifikation', n_samples=500, n_features=8)
# CSV-Datei im data/-Ordner speichern
df.to_csv('../data/synthetisches_datensatz_klassifikation.csv', index=False)
df.head()

In [ ]:
# Schnelle EDA
print('Form:', df.shape)
display(df.describe().T)

# Ziel-Verteilung (für Klassifikation sollte die Zielspalte 0/1 sein)
try:
    sns.countplot(x='ziel', data=df)
    plt.title('Verteilung der Zielvariablen')
    plt.show()
except Exception:
    print('Countplot konnte nicht erstellt werden')

# Korrelationsmatrix
corr = df.corr()
plt.figure(figsize=(8,6))
sns.heatmap(corr, cmap='coolwarm', center=0)
plt.title('Korrelationsmatrix (Merkmale + Ziel)')
plt.show()

In [ ]:
# Einfacher Pipeline-Workflow (Klassifikation)
X = df.drop(columns=['ziel'])
y = df['ziel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

pipeline = Pipeline([
    ('skalierer', StandardScaler()),
    ('modell', RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('Classification Report:')
print(classification_report(y_test, y_pred))

# Optional: ROC, Verwirrungsmatrix etc. können hier ergänzt werden